<a href="https://colab.research.google.com/github/piercegregory-spec/python-exercises/blob/main/benelux_cuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Benelux pairs (Erdős #850) on a GPU — `benelux_cuda.ipynb`

CUDA version of the large-range search in `benelux_seg.py` / `benelux.ipynb`, written for **Google Colab** (Runtime → Change runtime type → a GPU such as A100, L4 or T4). The notebook is self-contained: no other files are needed.

**Definitions.** rad(x) = product of the distinct primes dividing x. A *first-kind* pair is m < n with rad(m) = rad(n) and rad(m+1) = rad(n+1). A *second-kind* pair has rad(m) = rad(n+1) and rad(m+1) = rad(n). Known results: infinite families plus the exceptional pairs **(75, 1215)** (first kind) and **(35, 4374)** (second kind).

**Algorithm (same maths as the CPU script).** For each n, with r = rad(n) and s = rad(n+1):
1. n and n+1 must both be non-squarefree, otherwise n cannot be the larger member (this skips ~85% of integers).
2. The only possible partner m is fixed by the Chinese remainder theorem (m ≡ 0 mod r, m ≡ -1 mod s for first kind; roles swapped for second kind). Only if that m is below n is it worth checking.
3. The few survivors are verified exactly on the CPU by trial division.

**How the GPU is used.** Work is processed in blocks of B = 2^24 consecutive integers (≈16.8M). Per block, three kernels run:

| Kernel | One GPU thread = | Job |
|---|---|---|
| `k_mark` | one (prime, chunk of 256 multiples) | Segmented sieve: records each prime that divides each number, using an atomic counter to reserve a slot (no write conflicts). |
| `k_rad` | one number | Multiplies the recorded primes (plus any leftover large prime) to get rad(x). |
| `k_cand` | one n | Applies the squarefree filter and the CRT test; writes surviving candidates to a buffer. |

**Notes.**
- Colab gives one GPU at a time, so "multiple cores" here means the thousands of CUDA cores on that one GPU.
- Progress is checkpointed to `OUTDIR` (default `benelux_runs`). To keep it across Colab disconnects, mount Google Drive and point `OUTDIR` at a Drive folder (see the last cell). Re-running with the same `LO, HI, B` resumes.
- The kernels were checked for correctness only with Numba's CUDA *simulator* (against the CPU version); real-GPU speed has not been measured yet. Run the self-test cell first, then time a small range before committing to 10^12.
- Memory: about 1.2 GB of GPU RAM at B = 2^24. Lower `B` if you hit an out-of-memory error.


## 1. Check the GPU

In [ ]:
!nvidia-smi
import numba
from numba import cuda
print("numba", numba.__version__, "| CUDA available:", cuda.is_available())
if cuda.is_available():
    print(cuda.get_current_device().name)


## 2. Implementation (kernels + driver)

Defines the GPU kernels, the CPU verifier for the rare candidates, and `search_cuda(lo, hi, B, outdir)`, which returns `(first_kind, second_kind)` as sorted lists of `(m, n)`.

In [2]:
import math
import os
import time

import numpy as np
from numba import cuda, njit

NSLOT = 14      # max distinct prime factors per number we can record (<= 12 for n < 1e13)
K_CHUNK = 256   # multiples of one prime handled by one GPU thread
TPB = 128       # threads per CUDA block


def primes_upto(limit):
    sieve = np.ones(limit + 1, dtype=np.bool_)
    sieve[:2] = False
    for i in range(2, int(limit ** 0.5) + 1):
        if sieve[i]:
            sieve[i * i::i] = False
    return np.nonzero(sieve)[0].astype(np.int64)


# ---------------------------------------------------------------- GPU kernels
@cuda.jit(device=True)
def modinv_dev(a, m):
    """Inverse of a mod m (gcd 1, m >= 2); all intermediates stay <= m."""
    a = a % m
    if a == 0:
        return -1
    m0 = m
    x0 = 0
    x1 = 1
    while a > 1:
        q = a // m
        t = m
        m = a % m
        a = t
        t = x0
        x0 = x1 - q * x0
        x1 = t
    if x1 < 0:
        x1 += m0
    return x1


@cuda.jit
def k_mark(L, size, item_p, item_c, nitems, K, cnt, slots, flag):
    """One thread = (prime, chunk of its multiples). Records p in the slot list of
    every number in [L, L+size) that p divides (atomic add reserves the slot)."""
    t = cuda.grid(1)
    if t >= nitems:
        return
    p = np.int64(item_p[t])
    c = np.int64(item_c[t])
    j = ((L + p - 1) // p) * p - L + c * K * p
    for _ in range(K):
        if j >= size:
            break
        s = cuda.atomic.add(cnt, j, 1)
        if s < NSLOT:
            slots[s, j] = item_p[t]
        else:
            cuda.atomic.max(flag, 0, 1)
        j += p


@cuda.jit
def k_rad(L, size, cnt, slots, rad):
    """One thread = one number: rad = product of recorded primes (x contains
    each of them), times the leftover large prime factor (> sqrt(hi)) if any."""
    j = cuda.grid(1)
    if j >= size:
        return
    x = L + j
    r = x - x + 1
    n = cnt[j]
    if n > NSLOT:
        n = NSLOT
    for s in range(n):
        p = np.int64(slots[s, j])
        r *= p
        while x % p == 0:
            x //= p
    if x > 1:
        r *= x
    rad[j] = r
    cnt[j] = 0                      # reset for the next block


@cuda.jit
def k_cand(L, nn, rad, out, ocount, cap):
    """One thread = one n. Emits (m0, n, a, b, kind) for each CRT solution m0 < n."""
    i = cuda.grid(1)
    if i >= nn:
        return
    n = L + i
    r = rad[i]
    s = rad[i + 1]
    if r == n or s == n + 1:        # n or n+1 squarefree -> cannot be the larger member
        return
    for kind in range(1, 3):
        a = r if kind == 1 else s   # m = 0 (mod a), m = -1 (mod b)
        b = s if kind == 1 else r
        inv = modinv_dev(a, b)
        if inv < 0:
            continue
        k = (b - inv) % b
        if k == 0 or k > (n - 1) // a:
            continue
        idx = cuda.atomic.add(ocount, 0, 1)
        if idx < cap:
            out[idx, 0] = a * k
            out[idx, 1] = n
            out[idx, 2] = a
            out[idx, 3] = b
            out[idx, 4] = kind


# ------------------------------------------------- CPU verification (rare hits)
@njit
def rad_td(x, primes):
    r = 1
    for p in primes:
        if p * p > x:
            break
        if x % p == 0:
            r *= p
            while x % p == 0:
                x //= p
    if x > 1:
        r *= x
    return r


@njit
def verify(cands, primes, res):
    c = 0
    for q in range(cands.shape[0]):
        m = cands[q, 0]
        n = cands[q, 1]
        a = cands[q, 2]
        b = cands[q, 3]
        kind = cands[q, 4]
        step_ok = a <= n // b
        step = a * b if step_ok else 0
        while m < n:
            if rad_td(m, primes) == a and rad_td(m + 1, primes) == b:
                if c < res.shape[0]:
                    res[c, 0] = kind
                    res[c, 1] = m
                    res[c, 2] = n
                c += 1
            if not step_ok:
                break
            m += step
    return c


# ------------------------------------------------------------------- driver
def search_cuda(lo, hi, B=1 << 24, outdir=".", cand_cap=1 << 18, quiet=False,
                checkpoint_every=8):
    """Search n in [lo, hi]. Returns (first_kind, second_kind) lists of (m, n)."""
    lo = max(lo, 2)
    primes = primes_upto(math.isqrt(hi + 2) + 2)
    # static work list: (prime, chunk) pairs, independent of the block start L
    counts = (B + 1) // primes + 1
    nch = (counts + K_CHUNK - 1) // K_CHUNK
    item_p = np.repeat(primes, nch).astype(np.int32)
    offs = np.cumsum(nch) - nch
    item_c = (np.arange(int(nch.sum())) - np.repeat(offs, nch)).astype(np.int32)
    nitems = len(item_p)

    d_item_p = cuda.to_device(item_p)
    d_item_c = cuda.to_device(item_c)
    d_cnt = cuda.to_device(np.zeros(B + 1, dtype=np.int32))
    d_slots = cuda.device_array((NSLOT, B + 1), dtype=np.int32)
    d_rad = cuda.device_array(B + 1, dtype=np.int64)
    d_out = cuda.device_array((cand_cap, 5), dtype=np.int64)
    d_ocount = cuda.to_device(np.zeros(1, dtype=np.int64))
    d_flag = cuda.to_device(np.zeros(1, dtype=np.int32))

    os.makedirs(outdir, exist_ok=True)
    tag = f"cuda_{lo}_{hi}_{B}"
    prog_path = os.path.join(outdir, f"benelux_progress_{tag}.txt")
    res_path = os.path.join(outdir, f"benelux_results_{tag}.txt")
    nxt = lo
    if os.path.exists(prog_path):
        nxt = int(open(prog_path).read().strip())
        if not quiet:
            print(f"resuming from {nxt:,}")
    found = []
    if os.path.exists(res_path):
        for line in open(res_path):
            found.append(tuple(map(int, line.split())))

    res = np.zeros((4096, 3), dtype=np.int64)
    t0 = time.perf_counter()
    done0 = nxt
    pending = []
    L = nxt
    nblocks = 0
    while L <= hi:
        nn = min(B, hi + 1 - L)
        size = nn + 1
        g_items = (nitems + TPB - 1) // TPB
        k_mark[g_items, TPB](np.int64(L), size, d_item_p, d_item_c, nitems,
                             K_CHUNK, d_cnt, d_slots, d_flag)
        g_num = (size + TPB - 1) // TPB
        k_rad[g_num, TPB](np.int64(L), size, d_cnt, d_slots, d_rad)
        d_ocount.copy_to_device(np.zeros(1, dtype=np.int64))
        k_cand[(nn + TPB - 1) // TPB, TPB](np.int64(L), nn, d_rad, d_out,
                                            d_ocount, cand_cap)
        noc = int(d_ocount.copy_to_host()[0])
        if noc > cand_cap:
            raise RuntimeError(f"candidate buffer overflow ({noc} > {cand_cap}); raise cand_cap")
        if noc:
            cands = d_out[:noc].copy_to_host()
            c = verify(cands, primes, res)
            if c > len(res):
                raise RuntimeError("result buffer overflow")
            for q in range(c):
                pending.append((int(res[q, 0]), int(res[q, 1]), int(res[q, 2])))
        L += B
        nblocks += 1
        if nblocks % checkpoint_every == 0 or L > hi:
            if int(d_flag.copy_to_host()[0]):
                raise RuntimeError("a number had more than NSLOT distinct primes; raise NSLOT")
            with open(res_path, "a") as f:
                for k, m, n in pending:
                    f.write(f"{k} {m} {n}\n")
            with open(prog_path, "w") as f:
                f.write(str(min(L, hi + 1)))
            if not quiet:
                el = time.perf_counter() - t0
                rate = (min(L, hi + 1) - done0) / max(el, 1e-9)
                eta = (hi + 1 - min(L, hi + 1)) / max(rate, 1e-9)
                print(f"  {min(L, hi + 1):,}/{hi:,}  {rate/1e9:.2f}e9 n/s  "
                      f"eta {eta/60:.1f} min" + (f"  new: {pending}" if pending else ""),
                      flush=True)
            found.extend(pending)
            pending = []

    first = sorted({(m, n) for k, m, n in found if k == 1})
    second = sorted({(m, n) for k, m, n in found if k == 2})
    if lo <= 3 <= hi and (2, 3) not in second:   # degenerate pair, see benelux_seg.py
        second.append((2, 3))
        second.sort()
    return first, second


## 3. Self-test (run this first)

Compares the GPU results for n ≤ 200,000 against a simple brute-force reference. Both lines should print `True`.

In [ ]:
from collections import defaultdict
import tempfile


def naive(N):
    rad = [1] * (N + 3)
    for p in range(2, N + 3):
        if rad[p] == 1:
            for j in range(p, N + 3, p):
                rad[j] *= p
    keys = defaultdict(list)
    for m in range(1, N + 1):
        keys[(rad[m], rad[m + 1])].append(m)
    first, second = set(), set()
    for lst in keys.values():
        for i in range(len(lst)):
            for j in range(i + 1, len(lst)):
                first.add((lst[i], lst[j]))
    for m in range(1, N + 1):
        for n in keys.get((rad[m + 1], rad[m]), []):
            if m < n:
                second.add((m, n))
    return sorted(first), sorted(second)


N_TEST = 200_000
with tempfile.TemporaryDirectory() as d:
    g1, g2 = search_cuda(2, N_TEST, B=1 << 16, outdir=d, quiet=True)
r1, r2 = naive(N_TEST)
print("first kind  match:", g1 == r1, len(g1), len(r1))
print("second kind match:", g2 == r2 or g2 == sorted(set(r2) | {(2, 3)}), len(g2), len(r2))


## 4. Run

Running the next cell **prompts you for N**, the largest n to search (it checks every n from 2 up to N). Accepted formats: `1000000000`, `1,000,000,000`, `1e9`, `10**12`. Press Enter with no input for the default of 10^9. Try 10^9 first to measure speed, then scale up.

Choose **LO** the same way if you want to search a sub-range (e.g. to split a big run into pieces). Pairs not matching a known family are labelled `EXCEPTIONAL`; any besides (75, 1215), (35, 4374) and the trivial (2, 3) would be new.

### Running very large searches (10^12 and beyond)

**Measured so far (Colab A100):** 10^9 took ~1 s (~1.05e9 integers/s); 10^12 took roughly 19 min (~0.9e9 integers/s). GPU RAM use was only ~2.5 GB of 80 GB and system RAM ~3 GB, and neither grows with N, so memory is not the limit; only time is.

**Estimated cost of a 10^13 search:** ~3-4 hours, i.e. about 20-27 Colab compute units (Pro+ usage was ~6.77 units/hour). Update this with the real `elapsed` time from your own runs.

| N | Approx. time | Approx. compute units |
|---|---|---|
| 10^9 | ~1 s | ~0 |
| 10^11 | ~2 min | ~0.2 |
| 10^12 | ~19 min | ~2 |
| 10^13 | ~3-4 h | ~20-27 |
| 10^14 | ~30+ h | too long for one session; use slices |

For reference, Hercher's published search reached 1.4 x 10^12, so anything beyond that is new territory.

**Before a long run:**
1. **Mount Google Drive** so checkpoints survive a disconnect. In the cell below, uncomment the three `drive` / `OUTDIR` lines.
2. **Keep the browser tab open** and stay on a GPU runtime.
3. **To resume after a disconnect:** reconnect, re-run the implementation cell (section 2), then run the cell below with the **same start, N and `B`**. Checkpoints are keyed on those three values, so changing any of them starts fresh.

**Splitting a big search into slices.** Use the second prompt ("Start from n") to search only part of the range. The prompt accepts plain digits, commas, `1e12`-style and `10**12`-style input, but not expressions like `10**12 + 1`, so type start values as plain digits. Example plan for 10^13:
- Slice 1: start `2`, N `10**12` (done).
- Slice 2: start `1000000000001`, N `5e12`.
- Slice 3: start `5000000000001`, N `1e13`.

Each slice's results are saved in separate files, so you can run them in different sessions.

**Any `EXCEPTIONAL` result** other than (75, 1215), (35, 4374) and the trivial (2, 3) would be a new Benelux pair worth double-checking independently (for example by re-running that range with the CPU script `benelux_seg.py`).

In [ ]:
import re
import time
from decimal import Decimal

# --- optional: persist checkpoints on Google Drive ---------------------------
# from google.colab import drive
# drive.mount("/content/drive")
# OUTDIR = "/content/drive/MyDrive/benelux_runs"
OUTDIR = "benelux_runs"

B = 1 << 24           # integers per GPU block; keep the same value when resuming


def parse_count(text, default):
    """Accepts 1000000, 1,000,000, 1e9, 2.5e11 or 10**12. Empty -> default."""
    t = text.strip().replace(",", "").replace("_", "").replace(" ", "")
    if not t:
        return default
    if re.fullmatch(r"\d+", t):
        return int(t)
    m = re.fullmatch(r"(\d+)\*\*(\d+)", t)
    if m:
        return int(m.group(1)) ** int(m.group(2))
    m = re.fullmatch(r"(\d+(?:\.\d+)?)[eE](\d+)", t)
    if m:
        return int(Decimal(m.group(1)) * 10 ** int(m.group(2)))
    raise ValueError(f"Could not understand {text!r}; try 1000000000, 1e9 or 10**12")


HI = parse_count(input("Search up to N (e.g. 10**9, 1e12) [default 10**9]: "), 10**9)
LO = parse_count(input("Start from n (Enter for 2): "), 2)
if LO < 2 or HI < LO:
    raise ValueError(f"Need 2 <= start <= N, got start={LO:,}, N={HI:,}")
print(f"Searching n from {LO:,} to {HI:,}\n")


def is_pow2(x):
    return x > 0 and (x & (x - 1)) == 0


def classify_first(m, n):
    s = m + 2
    if is_pow2(s):
        k = s.bit_length() - 1
        if k >= 2 and n == 2 ** (2 * k) - 2 ** (k + 1):
            return f"infinite family (k={k})"
    return "EXCEPTIONAL"


def classify_second(m, n):
    s = m - 1
    if is_pow2(s):
        k = s.bit_length() - 1
        if k >= 0 and n == m * m - 1:
            return f"infinite family (k={k})"
    return "EXCEPTIONAL"


t0 = time.perf_counter()
p1, p2 = search_cuda(LO, HI, B=B, outdir=OUTDIR)
print(f"\nfirst kind: {len(p1)} pairs, second kind: {len(p2)} pairs")
for m, n in p1:
    print(f"  1st ({m}, {n})  [{classify_first(m, n)}]")
for m, n in p2:
    print(f"  2nd ({m}, {n})  [{classify_second(m, n)}]")
print("exceptional first :", [p for p in p1 if classify_first(*p) == "EXCEPTIONAL"])
print("exceptional second:", [p for p in p2 if classify_second(*p) == "EXCEPTIONAL"])
print(f"elapsed: {time.perf_counter() - t0:.1f}s")
